# STAR testing - Hamilton STAR / STARLet

Validation notebook for the v1 STAR (`pylabrobot.hamilton.star`).

It drives a `STARDevice` - the instrument as a resource, with its deck as its child. The driver
stays reachable underneath as `star.driver`, and machine-level reads that the device does not
proxy are sent through it.

**`setup()` moves the machine.** Watch the log: it reports each phase at `DEBUG` and the machine
it found at `INFO`. It runs in three steps:

1. **discover** - read-only. Machine configuration, arm geometry, channel count, and every
   channel's firmware, width and installed hardware.
2. **initialize** - `C0 VI` on a machine that is not initialized, which homes every drive; or
   `C0 ZA` alone on one that is, to raise the channels to Z safety.
3. **capability initialization** - the channels eject whatever is mounted on them, including grippers.

If you want to connect and look without anything moving, run `discover()` on its own; the cell
below shows how.

Set `protocol_mode` to `"simulation"` to run every cell against a simulated STAR - no hardware,
no USB, and the same code paths as the real driver.

## 1- Run identity

In [1]:
# --- Run identity ---
protocol_mode = "execution"  # simulation OR execution
user_name = "star_user"
run_identifier = "star_v1_validation"

# --- Which instrument ---
# One of the factories in pylabrobot.hamilton.star.device. It fixes the machine's footprint and
# where its deck sits inside it, and builds the matching deck.
instrument = "STAR"  # STAR OR STARLet OR STAR_with_extension_housing

# --- Device selection (only needed with more than one Hamilton on USB) ---
device_address = None  # USB address, e.g. 3
serial_number = None  # USB serial, e.g. "1234567"

# --- Motion ---
# The X-arm move and the 96-head sections only run when this is True.
allow_x_arm_move = True

# --- 96-head ---
# Where the 96-head ejects when it is initialized: head channel A1, in deck mm. Initializing it
# throws off whatever is mounted, so this has to be somewhere tips may be dropped, which depends
# on where the waste sits on this deck - hence no default. Setup initializes the head when this is
# set, and reports that it cannot when it is None. This machine was last sent (-263.8, 108.3,
# 200.0), read off the head's own initialization command in an earlier run.
head96_tip_discard_location = (-263.8, 108.3, 200.0)  # as a Coordinate below, after imports

# --- Tips ---
# Section 13 collects a rack of tips on the 96-head and puts them straight back. Off by default:
# tip handling is not what this driver is being validated for yet, and the section models a carrier
# onto the deck as soon as it runs, which would put the model out of step with a deck that has none.
# To use it, set this to the track a tip carrier is physically loaded on.
tip_carrier_track = None

## 2- Imports

In [2]:
from pylabrobot.hamilton.star.device import STAR, STAR_with_extension_housing, STARLet
from pylabrobot.hamilton.star.driver.master import STARDriver
from pylabrobot.hamilton.star.resource_model import NChannelPipette, TipMountingShaft
from pylabrobot.resources.coordinate import Coordinate
from pylabrobot.resources.hamilton import TIP_CAR_480_A00, hamilton_96_tiprack_1000uL

## 3- Logging

Uses PyLabRobot's own `setup_logger`, exactly as every other PLR run does: a single
date-stamped file per day, appended to across runs. Both the file and the notebook are at
`IO` level, so every byte sent to and received from the machine is visible and recorded.

Re-running this cell is safe: `setup_logger` replaces the file handler and `verbose`
replaces the console handler, rather than stacking a second one of each.

In [3]:
import logging

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO

log_dir = f"_logs/{protocol_mode}"

# PLR's own logger setup: one date-stamped file per day, appended to across runs. Re-running this
# cell replaces the file handler rather than stacking a second one, so lines are never duplicated.
pylabrobot.setup_logger(log_dir, level=LOG_LEVEL_IO)

# Console at IO level too: every byte sent and received appears in the notebook.
pylabrobot.verbose(True, level=LOG_LEVEL_IO)

print(f"appending to {log_dir}/pylabrobot-<YYYYMMDD>.log")
logging.getLogger("pylabrobot").info("--- %s (%s) ---", run_identifier, protocol_mode)

2026-08-26 18:36:52,817 - pylabrobot - INFO - --- star_v1_validation (execution) ---


appending to _logs/execution/pylabrobot-<YYYYMMDD>.log


## 4- Connect and bring the machine up

In simulation this is a `STARSimulationDriver`, which answers as a real instrument does - the
same command assembly, error decoding and response parsing run either way.

Set `head96_tip_discard_location` above for setup to initialize the 96-head too; without it, setup
brings everything else up and reports that it could not do the head.

To connect **without moving anything**, replace `await star.setup()` with:

```python
await star._open()
star._connected = True
await star.discover()
```

In [4]:
build = {
  "STAR": STAR,
  "STARLet": STARLet,
  "STAR_with_extension_housing": STAR_with_extension_housing,
}[instrument]

# The instrument builds its own deck and hands it to the driver, which models the machine into it.
# A simulated one answers from that model, so it is built here rather than passed in.
if protocol_mode == "execution":
  star = build(driver=STARDriver(device_address=device_address, serial_number=serial_number))
else:
  star = build(simulation=True)

# A capability cannot be configured before setup: it hangs off an arm, and the arms are not known
# until discovery has read the machine. So setup runs first and reports that it could not initialize
# the 96-head, and the cell below does that separately.
await star.setup()

print(star)
# setup logs this summary at INFO; printed here too so it is the first thing you see.
print(star.driver.format_setup_summary())

2026-08-26 18:36:52,835 - pylabrobot.hamilton.star.driver.master - DEBUG - Setting up STAR on USB 0x08af:0x8000 ...
2026-08-26 18:36:52,838 - pylabrobot.io.usb - INFO - Finding USB device...
2026-08-26 18:36:52,855 - pylabrobot.io.usb - INFO - Found USB device.
2026-08-26 18:36:52,866 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-08-26 18:36:55,871 - pylabrobot.hamilton.star.drive

Hamilton STAR(STARDriver, 56-track deck)
[Hamilton STAR] Connected on USB 0x08af:0x8000
  Firmware: master 7.6S 25 2021_11_05 (GRU C0), pipettes 4.0S j 2022-03-16, x_arm 1.4S 2012-04-25, head96 5.0S i 2021-10-22 (H0 XE167), iswap 4.1S 2011-12-19, autoload 3.4S f 2017-01-09
  Configuration: 54 slots
  Autoload: 1D barcode scanner
  Arms: 1
    left: hamilton_legacy_star_dual_rail_arm, 354.0 mm wide, travel 95.0 to 1340.2 mm, workspace -323.2 to 1517.2 mm
      channels: 8 (1000uL) | 96-head: 96 head II | 384-head: none | iSWAP: wide gripper


In [5]:
deck = star.deck
deck

HamiltonSTARDeck(name='deck', location=Coordinate(110.000, 097.800, 078.500), size_x=1545, size_y=653.5, size_z=900, category=deck)

In [6]:
star.driver.deck

HamiltonSTARDeck(name='deck', location=Coordinate(110.000, 097.800, 078.500), size_x=1545, size_y=653.5, size_z=900, category=deck)

### Initializing the 96-head

**This descends.** The head ejects whatever is mounted when it initializes, at
`head96_tip_discard_location`, which is machine-specific and cannot be checked from here. Gated on
`allow_head96_initialize`, set in the cell itself.

Setup retracts the head either way, so everything below runs whether or not this does.

In [7]:
allow_head96_initialize = False

# Initializing the 96-head makes it descend to `head96_tip_discard_location` and eject whatever is
# mounted, so it is separate from setup and off by default: where a head ejects is machine-specific
# and nothing here can check it. Setup leaves the head retracted either way, so the sections below
# work whether or not this runs - unless the head refuses to move while it reports itself
# uninitialized, which is what this is here for.
if star.head96 is None:
  print("no 96-head on this machine")
elif head96_tip_discard_location is None:
  print("no head96_tip_discard_location set at the top of the notebook")
elif await star.driver.request_initialization_status(star.head96.configuration.module):
  print("the 96-head reports itself initialized already; nothing to do")
elif not allow_head96_initialize:
  print(
    f"skipped. the head reports itself uninitialized. set allow_head96_initialize = True to send "
    f"it to {head96_tip_discard_location} and eject"
  )
else:
  star.head96.configuration.tip_discard_location = Coordinate(*head96_tip_discard_location)
  await star.head96.initialize()
  print(f"initialized; the head ejected at {head96_tip_discard_location}")

2026-08-26 18:37:56,373 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0QWid0081'
2026-08-26 18:37:56,380 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0QWid0081qw0')


skipped. the head reports itself uninitialized. set allow_head96_initialize = True to send it to (-263.8, 108.3, 200.0) and eject


## 5- What setup found

Everything below depends on what discovery read, so this prints it in full before anything uses it.

The last table is the one to look at. The machine's configuration says what is installed; the driver
builds a capability for each of those. A row where the two disagree means discovery declined to
build something the machine claims - which is either a bug or a bit set on a machine that no longer
carries the part.

In [8]:
print(star.driver.format_setup_summary())

c = star.driver.configuration
print(f"\nchannels    : {star.driver.num_channels}")
print(f"instrument  : {c.instrument_size_slots} slots")

print("\nfirmware, per module:")
for module, version in star.driver.firmware.items():
  print(f"  {module:<12} {version}")

# What the configuration claims, against what the driver built for it.
rows = [
  ("autoload", c.autoload_installed, star.driver.autoload),
  ("front cover", c.main_front_cover_monitoring_installed, star.front_cover),
]
for arm in star.driver.arms:
  a = arm.configuration
  rows += [
    (f"{arm.side} channels", a.pip_installed, arm.pipettes),
    (f"{arm.side} 96-head", a.head96_installed, arm.head96),
    (f"{arm.side} 384-head", a.head384_installed, arm.head384),
    (f"{arm.side} iSWAP", a.iswap_installed, arm.iswap),
  ]

print(f"\n  {'capability':<20} {'installed':<11} {'built'}")
for label, installed, built in rows:
  disagree = "   <- disagree" if bool(installed) != (built is not None) else ""
  print(
    f"  {label:<20} {str(bool(installed)):<11} {'yes' if built is not None else 'no'}{disagree}"
  )

[Hamilton STAR] Connected on USB 0x08af:0x8000
  Firmware: master 7.6S 25 2021_11_05 (GRU C0), pipettes 4.0S j 2022-03-16, x_arm 1.4S 2012-04-25, head96 5.0S i 2021-10-22 (H0 XE167), iswap 4.1S 2011-12-19, autoload 3.4S f 2017-01-09
  Configuration: 54 slots
  Autoload: 1D barcode scanner
  Arms: 1
    left: hamilton_legacy_star_dual_rail_arm, 354.0 mm wide, travel 95.0 to 1340.2 mm, workspace -323.2 to 1517.2 mm
      channels: 8 (1000uL) | 96-head: 96 head II | 384-head: none | iSWAP: wide gripper

channels    : 8
instrument  : 54 slots

firmware, per module:
  master       7.6S 25 2021_11_05 (GRU C0)
  pipettes     4.0S j 2022-03-16
  x_arm        1.4S 2012-04-25
  head96       5.0S i 2021-10-22 (H0 XE167)
  iswap        4.1S 2011-12-19
  autoload     3.4S f 2017-01-09

  capability           installed   built
  autoload             True        yes
  front cover          False       no
  left channels        True        yes
  left 96-head         True        yes
  left 384-head     

## 6- The resource model

What the machine carries is modelled as resources on the deck, placed where the drives say they are:
each arm, every pipetting channel, each head, the autoload sled. Nothing here moves or reads the
machine - it prints what setup already built.

The tip mounting shafts are the part that matters for tips. Each pipetting channel and each head
channel carries one, and a shaft is what a collected tip is meant to hang from - so the tree itself
would answer what is mounted, with no separate record to keep in step with it.

`mount_tip` and `release_tip` are on the shaft, but nothing calls them yet: collecting tips does not
put them on the model. The count below therefore reads zero whatever the head is carrying, and
section 13 is where that shows.

In [9]:
def show(resource, indent=0, max_depth=3, max_siblings=4):
  """Print a resource and its children, abbreviating wide and deep branches."""
  print(f"{'  ' * indent}{resource.name}  ({type(resource).__name__})")
  if indent >= max_depth:
    if resource.children:
      print(f"{'  ' * (indent + 1)}... {len(resource.children)} children")
    return
  for child in resource.children[:max_siblings]:
    show(child, indent + 1, max_depth, max_siblings)
  if len(resource.children) > max_siblings:
    print(f"{'  ' * (indent + 1)}... {len(resource.children) - max_siblings} more")


show(star)

children = star.get_all_children()
shafts = [r for r in children if isinstance(r, TipMountingShaft)]
print(f"\n{len(shafts)} tip mounting shafts, {sum(s.has_tip() for s in shafts)} carrying a tip")

for pipette in [r for r in children if isinstance(r, NChannelPipette)]:
  print(
    f"\n{pipette.name}: {pipette.num_channels} channels, {pipette.channel_pitch} mm pitch, "
    f"{pipette.tip_pickup_mode} pickup, independent actuation "
    f"{pipette.independent_channel_actuation}"
  )
  a1 = pipette.get_item("A1")
  print(f"  A1 at {a1.get_location_wrt(deck)} in deck mm")

Hamilton STAR  (STARDevice)
  deck  (HamiltonSTARDeck)
    trash_core96  (Trash)
    waste_block  (Resource)
      teaching_tip_rack  (TipRack)
        ... 8 children
      core_grippers  (HamiltonCoreGrippers)
    trash  (Trash)
    left_x_arm  (Resource)
      pipette_channel_0  (Resource)
        ... 1 children
      pipette_channel_1  (Resource)
        ... 1 children
      pipette_channel_2  (Resource)
        ... 1 children
      pipette_channel_3  (Resource)
        ... 1 children
      ... 5 more
    ... 2 more

104 tip mounting shafts, 0 carrying a tip

head96: 96 channels, 9.0 mm pitch, core pickup, independent actuation False
  A1 at Coordinate(021.600, 554.800, 336.970) in deck mm


## 7- Has this firmware stack been driven before?

What the machine is made of is captured in full by the capture script; what a run wants to know here
is whether any of its boards report firmware this driver has not been driven against.


In [10]:
from pylabrobot.hamilton.star.driver.confirmed_firmware_versions import suggest_entry, unconfirmed

# Has each of this machine's boards been driven on the firmware it reports?
new = unconfirmed(star.driver.firmware)
print()
if not new:
  print(f"firmware: all {len(star.driver.firmware)} capabilities confirmed")
else:
  print(f"firmware: {len(new)} of {len(star.driver.firmware)} capabilities not seen before.")
  print("if this machine works, add them to confirmed_firmware_versions.py:")
  for capability, version in new.items():
    print(suggest_entry(capability, version))


firmware: all 6 capabilities confirmed


## 8- Sensor read: tip presence

Each channel's sleeve sensor reports whether a tip is mounted. This reads sensors; it does not
move a channel. After a full setup every channel should be empty: the channel initialization
ejects whatever was on them.

In [11]:
presence = await star.driver.request_tip_presence()
for channel, has_tip in enumerate(presence):
  print(f"  channel {channel}: {'tip' if has_tip else '-'}")

2026-08-26 18:37:56,431 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RTid0082'
2026-08-26 18:37:56,475 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RTid0082er00/00rt0 0 0 0 0 0 0 0')


  channel 0: -
  channel 1: -
  channel 2: -
  channel 3: -
  channel 4: -
  channel 5: -
  channel 6: -
  channel 7: -


## 9- The front cover

Two read-only commands, and what they mean is exactly what this check is for.

`C0 RW` reports three inputs, the first of them the cover input. `C0 QC` reports the cover
position. Neither says whether a cover is *fitted*: the master acts only on its non-volatile
configuration, so `main_front_cover_monitoring_installed` is what decides whether the cover is
watched at all, and `star.front_cover` exists only when it is set.

This machine reports it as not installed while the cover and its switch are physically there, so
`QC` is sent raw below rather than through the capability.

**Run this three times and record what changes**: cover shut, cover open, and cover cable
disconnected. If the cover input tracks the position it is a position input; if it holds while
the position changes it is a presence input; if neither moves, the master is not reading the
switch at all - which is what a configuration that says the monitoring is not installed predicts.

That last outcome is the one that decides whether `FrontCover` is worth keeping: a machine that
answers nothing here has no cover to drive, and the capability would only ever be an empty
`request_position` on machines configured differently from this one.


In [12]:
cover_input, second_input, reserve_input = await star.driver.request_cover_input_status()
print(f"inputs        : cover={cover_input}  second={second_input}  reserve={reserve_input}")

c = star.driver.configuration
print(
  f"monitoring    : main={c.main_front_cover_monitoring_installed}"
  f"  additional={c.additional_front_cover_monitoring_installed}"
)
print(f"covers        : left={c.left_cover_installed}  right={c.right_cover_installed}")
print(f"capability    : {star.front_cover}")

# C0 QC - request cover position. Read-only, and sent raw so it answers even on a machine whose
# configuration says the monitoring is not installed.
print(f"position (raw): {await star.driver.send_raw_command('C0QCid9989')}")
if star.front_cover is not None:
  print(f"position      : {await star.front_cover.request_position()}")

2026-08-26 18:37:56,485 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RWid0083'
2026-08-26 18:37:56,507 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RWid0083er00/00rw000')
2026-08-26 18:37:56,510 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0QCid9989'
2026-08-26 18:37:56,529 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0QCid9989er00/00qc1')


inputs        : cover=False  second=False  reserve=False
monitoring    : main=False  additional=False
covers        : left=False  right=False
capability    : None
position (raw): C0QCid9989er00/00qc1


## 10- Move the X-arm

**This moves the arm and everything mounted on it.** Only run it with the deck clear along the
path, and only after setup has raised the channels to Z safety.

Gated on `allow_x_arm_move`, set at the top of the notebook.

In [13]:
arm = star.x_arm
print(f"travel range: {arm.configuration.x_range} mm")

target = 500.0
if allow_x_arm_move:
  await arm.move_x(target)
  print(f"moved to {target} mm")
else:
  print(f"skipped. set allow_x_arm_move = True to move to {target} mm")

# out-of-range targets are refused before anything reaches the wire
try:
  await arm.move_x(5000.0)
except ValueError as e:
  print("guard:", e)

travel range: (95.0, 1340.2) mm


2026-08-26 18:37:56,540 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0084la05000lr3lw7'
2026-08-26 18:37:57,902 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0084er00')
2026-08-26 18:37:57,907 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0085'
2026-08-26 18:37:57,916 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0085rx+04999 +000049990')
2026-08-26 18:37:57,919 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0086'
2026-08-26 18:37:57,930 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0086rx+04999 +000049993')
2026-08-26 18:37:57,933 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0087'
2026-08-26 18:37:57,942 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0087rx+04999 +000049994')
2026-08-26 18:37:57,945 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0088'
2026-08-26 18:37:57,955 - pylabrobot.io.usb - IO - [0x8af:0x8000][]

moved to 500.0 mm
guard: left X-arm x=5000.0mm is outside its travel range [95.0, 1340.2].


In [14]:
# The same gate as the section above: this moves the arm. It reads the gate rather than setting
# it - setting it here would leave every later section unlocked too.
if allow_x_arm_move:
  await star.x_arm.move_x(500.0)
  print("moved to 500.0 mm")
else:
  print("skipped the move. the comparison below runs at wherever the arm is now")

# What the machine says, and where the model puts the arm's reference point. They should agree.
position = await star.x_arm.request_position()
arm_resource = deck.get_resource("left_x_arm")
seated = arm_resource.get_location_wrt(deck)
print(f"machine: {position} mm")
print(f"model  : {seated.x + arm_resource.get_anchor(x=star.x_arm.reference_anchor).x} mm")

2026-08-26 18:37:58,004 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0091la05000lr3lw7'
2026-08-26 18:37:58,212 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0091er00')
2026-08-26 18:37:58,216 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0092'
2026-08-26 18:37:58,225 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0092rx+04999 +000049994')
2026-08-26 18:37:58,229 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0093'
2026-08-26 18:37:58,238 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0093rx+04999 +000049994')
2026-08-26 18:37:58,242 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0094'
2026-08-26 18:37:58,251 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0094rx+04999 +000049994')
2026-08-26 18:37:58,255 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0095'
2026-08-26 18:37:58,264 - pylabrobot.io.usb - IO - [0x8af:0x8000][]

moved to 500.0 mm
machine: 500.0 mm
model  : 500.0 mm


## 11- The 96-head: park

Park is two moves, not the head's own home command: up to safe Z first, then across in Y to the
first of the Y positions the head has stored. In that order because the head crosses the deck to get
there - raised it sweeps over what is loaded, low it sweeps through it.

Only the order has been checked, in simulation. This is the first time the machine runs it.

**This moves the 96-head** in Z and then in Y. The deck has to be clear along its sweep. Gated on
`allow_x_arm_move`.

In [15]:
if not allow_x_arm_move:
  print("skipped. set allow_x_arm_move = True to move the head")
elif star.head96 is None:
  print("no 96-head on this machine")
else:
  head = star.head96
  print(
    f"before : y={await head.request_y_position():7.2f}  z={await head.request_z_position():7.2f}"
  )

  stored = await head.request_predefined_y_positions()
  print(f"stored y positions: {stored}, so park is {stored[0]:.2f} mm")

  parked_at = await head.park()
  y_now, z_now = await head.request_y_position(), await head.request_z_position()
  print(f"after  : y={y_now:7.2f}  z={z_now:7.2f}, park() reported {parked_at:.2f}")

  # The head is a resource too, and park moved it. The model should have followed.
  if head.resource is not None:
    print(f"model  : {head.resource.get_item('A1').get_location_wrt(deck)}")

2026-08-26 18:37:58,346 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0100'
2026-08-26 18:37:58,355 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0100ry+35512 +35507')
2026-08-26 18:37:58,360 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RZid0101'
2026-08-26 18:37:58,369 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RZid0101rz+67401 +67393')
2026-08-26 18:37:58,374 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0102rapy'
2026-08-26 18:37:58,388 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0102py+35485 +35000 +35000 +35000 +35000 +35000 +35000 +35000 +35000 +35000')
2026-08-26 18:37:58,392 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0103razv'
2026-08-26 18:37:58,402 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0103zv17000')
2026-08-26 18:37:58,406 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0104razr'
2026-08-26 18:37:58,4

before : y= 554.80  z= 336.97
stored y positions: [554.45, 546.88, 546.88, 546.88, 546.88, 546.88, 546.88, 546.88, 546.88, 546.88], so park is 554.45 mm


2026-08-26 18:37:58,525 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0ZAid0105er00')
2026-08-26 18:37:58,529 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RZid0106'
2026-08-26 18:37:58,538 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RZid0106rz+67394 +67393')
2026-08-26 18:37:58,543 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RZid0107'
2026-08-26 18:37:58,551 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RZid0107rz+67394 +67393')
2026-08-26 18:37:58,554 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0108rapy'
2026-08-26 18:37:58,569 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0108py+35485 +35000 +35000 +35000 +35000 +35000 +35000 +35000 +35000 +35000')
2026-08-26 18:37:58,571 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0109rayv'
2026-08-26 18:37:58,581 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0109yv25000')
2026-08-26 

after  : y= 554.55  z= 336.96, park() reported 554.45
model  : Coordinate(158.800, 554.550, 336.960)


## 12- The 96-head: the Y floor

This head refuses to travel as far forward as its Y parameter accepts. The rig settled where the
real limit is (section 17) and the driver now carries it, so what is left to check is that the guard
and the machine still agree.

Three probes. The driver should refuse a target below the floor without anything reaching the wire;
the machine should accept the floor itself; and the drive should report it arrived there.

**The second probe moves the 96-head** forward in Y, to the front of its permitted area. It is
raised to safe Z first. Gated on `allow_x_arm_move`.

In [16]:
if star.head96 is None:
  print("no 96-head on this machine")
else:
  head = star.head96
  floor, ceiling = head.configuration.y_range
  print(f"permitted Y area: {floor:.3f} to {ceiling:.3f} mm")

  # 1- the driver refuses below the floor, before anything reaches the wire. No motion.
  try:
    await head.move_y(floor - 0.5)
    print(f"  {floor - 0.5:7.3f} mm  NOT REFUSED - the guard is not holding")
  except ValueError as error:
    print(f"  {floor - 0.5:7.3f} mm  refused by the driver: {error}")

  # 2- and the machine takes the floor itself.
  if not allow_x_arm_move:
    print("  skipped the move. set allow_x_arm_move = True to send the head to the floor")
  else:
    await head.move_to_safe_z()
    await head.move_y(floor)
    arrived = await head.request_y_position()
    print(f"  {floor:7.3f} mm  accepted, the drive reports {arrived:7.3f} mm")
    print(f"  the two agree: {abs(arrived - floor) < 0.05}")
    await head.park()
    print(f"  parked at {await head.request_y_position():.2f} mm")

permitted Y area: 102.000 to 562.500 mm
  101.500 mm  refused by the driver: y must be between 102.0 and 562.5, is 101.5


2026-08-26 18:37:58,828 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0115razv'
2026-08-26 18:37:58,837 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0115zv17000')
2026-08-26 18:37:58,841 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0116razr'
2026-08-26 18:37:58,850 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0116zr080000')
2026-08-26 18:37:58,853 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0ZAid0117za67394zv17000zr080000zw15'
2026-08-26 18:37:58,965 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0ZAid0117er00')
2026-08-26 18:37:58,969 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RZid0118'
2026-08-26 18:37:58,979 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RZid0118rz+67394 +67392')
2026-08-26 18:37:58,983 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RZid0119'
2026-08-26 18:37:58,992 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] rea

  102.000 mm  accepted, the drive reports 102.090 mm
  the two agree: False


2026-08-26 18:38:01,215 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0132yr35000')
2026-08-26 18:38:01,219 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0YAid0133ya35485yv25000yr35000yw15'
2026-08-26 18:38:03,180 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0YAid0133er00')
2026-08-26 18:38:03,185 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0134'
2026-08-26 18:38:03,194 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0134ry+35485 +35482')
2026-08-26 18:38:03,199 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0135'
2026-08-26 18:38:03,208 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0135ry+35485 +35482')


  parked at 554.41 mm


## 13- The 96-head: tips

The head collects a whole rack at once - it is rigid, so there is no per-channel selection - and this
picks a rack up and puts it straight back on the same rack. Nothing is thrown away.

Three checks, and one of them is expected to fail:

- **the commands** - the head should build the same collect and drop commands the legacy driver did
- **the sleeve sensors** - 96 channels should report a tip after the collect and none after the drop
- **the model** - the shafts should carry 96 tips, and they will not. Nothing calls `mount_tip` yet,
  so this reads zero both times. It is here to measure the gap, not to pass.

Where a discard would go is printed but not sent: the head's configured trash resolves when no
location is given, and that resolution is worth seeing without binning a rack of tips to see it.

**This moves the 96-head** in X, Y and Z, over the tip carrier. A tip carrier has to be loaded on
`tip_carrier_track`, with a full rack on its frontmost site. Gated on `allow_head96_tip_moves`, set
in the cell itself so running the notebook top to bottom does not reach for tips.

In [17]:
allow_head96_tip_moves = True

if star.head96 is None:
  print("no 96-head on this machine")
elif tip_carrier_track is None:
  print("no tip_carrier_track set at the top of the notebook")
else:
  head = star.head96

  # The carrier as it is physically loaded, so the rack's own geometry decides where the head goes.
  if "tip_carrier" in [child.name for child in deck.children]:
    carrier = deck.get_resource("tip_carrier")
    rack = deck.get_resource("tip_rack")
  else:
    carrier = TIP_CAR_480_A00(name="tip_carrier")
    rack = hamilton_96_tiprack_1000uL(name="tip_rack")
    carrier[0] = rack
    deck.assign_child_resource(carrier, rails=tip_carrier_track)

  a1 = rack.get_item("A1").get_location_wrt(deck)
  floor, ceiling = head.configuration.y_range
  print(f"rack A1 sits at {a1}")
  print(f"the head's permitted Y area is {floor:.2f} to {ceiling:.2f} mm")
  if not floor <= a1.y <= ceiling:
    raise RuntimeError(
      f"the head cannot reach the rack: its A1 is at y={a1.y:.2f} mm, outside {floor:.2f} to "
      f"{ceiling:.2f}. Load the carrier further back, or use a site further back on it."
    )

  # Where a discard would go, resolved the way discard_tips resolves it. Nothing is sent.
  configured = head.configuration.tip_discard_location
  print(f"a discard with no location would go to {configured}")

  if not allow_head96_tip_moves:
    print("\nskipped the moves. set allow_head96_tip_moves = True to collect and return a rack")
  else:
    shafts = [r for r in star.get_all_children() if isinstance(r, TipMountingShaft)]

    await head.pick_up_tip_rack(rack)
    print(
      f"\ncollected. sensors: {sum(await star.driver.request_tip_presence())} channels report a tip"
    )
    print(
      f"           model  : {sum(s.has_tip() for s in shafts)} shafts carry one (expected 0 - not wired)"
    )
    print(f"           rack   : {sum(s.has_tip() for s in rack.get_all_items())} spots still full")

    await head.drop_tips(rack)
    print(
      f"\nreturned.  sensors: {sum(await star.driver.request_tip_presence())} channels report a tip"
    )
    print(
      f"           model  : {sum(s.has_tip() for s in shafts)} shafts carry one (expected 0 - not wired)"
    )
    print(f"           rack   : {sum(s.has_tip() for s in rack.get_all_items())} spots full again")

    await head.park()

no tip_carrier_track set at the top of the notebook


## 14- The 384-head

Nothing on this machine, which is the point: the driver has to say so rather than assume a head that
is not there. On a machine that does carry one, this reads what it says about itself - all read-only,
nothing moves.

In [18]:
head384 = star.head384
if head384 is None:
  installed = any(arm.configuration.head384_installed for arm in star.driver.arms)
  print(f"no 384-head built. any arm reports one installed: {installed}")
else:
  cfg = head384.configuration
  print(f"type              : {cfg.head_type}")
  print(
    f"channels          : {cfg.channel_columns} x {cfg.channel_rows}, {cfg.channel_pitch} mm pitch"
  )
  print(f"permitted Y area  : {cfg.y_range[0]:.2f} to {cfg.y_range[1]:.2f} mm")
  print(
    f"permitted Z area  : {cfg.z_range_documented[0]:.2f} to {cfg.z_range_documented[1]:.2f} mm"
  )
  # Both are the head type's to decide, so they refuse until it has been read.
  try:
    print(f"dispensing drive  : {cfg.dispensing_drive_uL_per_increment:.9f} uL per increment")
    print(f"squeezer drive    : {cfg.squeezer_drive_mm_per_increment:.9f} mm per increment")
  except RuntimeError as error:
    print(f"drive resolutions : {error}")
  print(f"absolute cLLD     : {cfg.supports_lld_absolute_threshold_check}")
  print(
    f"\nthe drives report: y={await head384.request_y_position():.2f}  "
    f"z={await head384.request_z_position():.2f}"
  )

no 384-head built. any arm reports one installed: False


## 15- The iSWAP

Read-only. Every value below is one the arm keeps and reports; none of it moves the gripper.

The two link lengths and the rotation drive's X offset are what a kinematic model of the arm needs,
and until now the driver took them on faith. This is the check that the arm answers them, and that
what it answers is the geometry we assumed.

In [19]:
iswap = star.iswap
if iswap is None:
  installed = any(arm.configuration.iswap_installed for arm in star.driver.arms)
  print(f"no iSWAP built. any arm reports one installed: {installed}")
else:
  print(f"firmware          : {await iswap.request_firmware_version()}")
  print(f"link 1            : {await iswap.request_link_1_length():.2f} mm")
  print(f"link 2            : {await iswap.request_link_2_length():.2f} mm")
  print(f"rotation X offset : {await iswap.request_rotation_drive_x_offset():.2f} mm")
  print(f"rotation drive    : {await iswap.request_rotation_drive_positions()}")
  print(f"wrist drive       : {await iswap.request_wrist_drive_positions()}")
  print(f"Y positions       : {await iswap.request_y_positions()}")
  print(
    f"gripper           : {'wide' if star.driver.configuration.iswap_gripper_wide else 'small'}"
  )

2026-08-26 18:38:03,312 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'R0RFid0136'
2026-08-26 18:38:03,322 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'R0RFid0136rf4.1S 2011-12-19')
2026-08-26 18:38:03,324 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'R0RAid0137rapw'
2026-08-26 18:38:03,346 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'R0RAid0137pw+13000 -29007 +00156 +29068 +29500 +29068 +29068 +29068 +29068 +01378')
2026-08-26 18:38:03,347 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'R0RAid0138rapt'


firmware          : 4.1S 2011-12-19
link 1            : 137.80 mm


2026-08-26 18:38:03,375 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'R0RAid0138pt-26577 -26577 -08860 +09044 +26858 -26577 -26577 -26577 -26577 +01377')
2026-08-26 18:38:03,378 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RAid0139rakg'
2026-08-26 18:38:03,401 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RAid0139er00/00kg328')
2026-08-26 18:38:03,404 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'R0RAid0140rapw'


link 2            : 137.70 mm
rotation X offset : 32.80 mm


2026-08-26 18:38:03,426 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'R0RAid0140pw+13000 -29007 +00156 +29068 +29500 +29068 +29068 +29068 +29068 +01378')
2026-08-26 18:38:03,430 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'R0RAid0141rapt'


rotation drive    : {'home': 13000, 'left': -29007, 'front': 156, 'right': 29068, 'parking': 29500, 'extra_1': 29068, 'extra_2': 29068, 'extra_3': 29068, 'extra_4': 29068}


2026-08-26 18:38:03,456 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'R0RAid0141pt-26577 -26577 -08860 +09044 +26858 -26577 -26577 -26577 -26577 +01377')
2026-08-26 18:38:03,458 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'R0RAid0142rapy'
2026-08-26 18:38:03,488 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'R0RAid0142py+09855 +07000 +09000 +13550 +12600 +09855 +09855 +09855 +09855 +09855')


wrist drive       : {'home': -26577, 'right': -26577, 'straight': -8860, 'left': 9044, 'reverse': 26858, 'extra_1': -26577, 'extra_2': -26577, 'extra_3': -26577, 'extra_4': -26577}
Y positions       : {'home': 456.3, 'lower_limit': 324.1, 'upper_limit': 416.7, 'parking': 627.4, 'pre_parking': 583.4, 'extra_1': 456.3, 'extra_2': 456.3, 'extra_3': 456.3, 'extra_4': 456.3, 'extra_5': 456.3}
gripper           : wide


## 16- Error decoding

An error comes back as a code on a module, and the driver turns it into a named exception with the
module that raised it and what it was doing. Two ways in, and both are checked here.

The guards first: a target outside a drive's range never reaches the wire at all, so the machine
never gets a chance to refuse it. Then a real one - a parameter the master does not know, which is
read-only and produces a genuine error to decode without moving anything.

In [20]:
# 1- refused locally, nothing sent
try:
  await star.x_arm.move_x(5000.0)
except ValueError as error:
  print(f"local guard : {type(error).__name__}: {error}")

# 2- refused by the machine. Reading a parameter the master does not know is read-only.
try:
  await star.driver.send_command(module="C0", command="RA", ra="zz")
  print("machine     : no error - the master accepted an unknown parameter name")
except Exception as error:  # noqa: BLE001 - what it raises is exactly what is being checked
  print(f"machine     : {type(error).__name__}: {error}")
  errors = getattr(error, "errors", None)
  if errors:
    for module, decoded in errors.items():
      print(f"  {module}: {decoded}")

local guard : ValueError: left X-arm x=5000.0mm is outside its travel range [95.0, 1340.2].


2026-08-26 18:38:03,505 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RAid0143razz'
2026-08-26 18:38:03,528 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RAid0143er01/31')


machine     : STARFirmwareError: {'Master': CommandSyntaxError('Unknown parameter')}, C0RAid0143er01/31
  Master: 


## 17- What the rig settled, and what the driver does about it

These were open questions; the sections that answered them are gone, and what they found is in the
driver. Kept here so nobody re-derives them.

**An X move always arrives.** Every condition, every repeat, ends exactly on target. What looked
like a 0.2 to 0.4 mm error was a position read taken before the arm had stopped: the move's reply
comes when the move ends. `move_x` now reads until two reads in a row find the arm at the target.

**How the arm gets there depends on the acceleration.** At index 3 it approaches and never
overshoots; at index 5 it swings past and comes back, in five runs of five. A 96-head parked
forward makes the swing bigger - 0.42 against 0.32 mm - which is the arm carrying an off-centre
mass. It all settles in 27 to 90 ms.

**The master and the X-drive board agree** on where the arm is, so reading `X0 RX` is equivalent to
reading `C0 RX`, and neither `X0 XP` nor `C0 JX` closes a position loop the other does not.

**The 96-head's permitted Y area starts at 102.000 mm**, not the 93.75 mm its own parameter accepts.
Found by bisection, and it is exact: 6528 increments, to the increment. Nothing stores it - all 499
readable parameters across the head and the master were swept and none holds it - so it is compiled
into the firmware and the driver carries it as a measured constant. What it costs: on the frontmost
site of a standard tip carrier the head's A1 reaches rows A to E, and misses F by 1.2 mm.

**The autoload's scanner resolves 0.1 mm per increment**, measured as exactly 22.5 mm per track
between tracks 10 and 20, and confirmed by the unit's own configuration, which discovery now reads
rather than assuming. Its drive counts from track 1, a hundred millimetres along the deck: at track
10 it reads 202.5 mm, exactly nine tracks, so its zero sits on track 1 itself rather than half a
track off it.

**The front cover is watched by an input, not by `C0 QC`.** With the monitoring bit clear, `C0 RW`
char 1 followed the cover - 1 shut, 0 open - while `QC` answered "closed" both times. Whether `QC`
works on a machine that declares the monitoring is still open, and needs the configuration write in
section 18.

## 18- The instrument configuration, and what writing it would mean

`C0 AK` writes the machine's non-volatile configuration. It takes **21 parameters**, each with a
default, and the master's convention is that an unsent parameter takes its default - so a partial
`AK` does not change one field, it rewrites all of them. Sending `kb` alone would declare no
channels, no 96-head, a different arm width and a different waste position.

Every one of the 21 is readable: `C0 RM` answers `kb` and `kp`, `C0 QM` the other 19. The cell below
reads them, rebuilds the command that would restore exactly what the machine says today, and shows
what changes if the front cover monitoring bit is set. **It sends nothing.**

Why we would want to: with `kb` bit 2 clear, `C0 QC` answered `qc1` with the cover open, so the
master is not reading the switch. Setting the bit is the only way to find out whether `QC` reports
the cover on a machine that declares the monitoring - and it is also what makes the machine abort a
run when the cover opens, which is why it was turned off in the first place.


In [21]:
import re

# The 21 parameters AK takes, in the order the specification lists them.
AK_PARAMETERS = "ka ke xt xa xw kb xl xn xr xo xm xx xu xv kp ys kl km ym yu yx".split()


def read_fields(reply: str) -> dict:
  """The two-letter fields in a reply, as the machine wrote them."""
  return dict(re.findall(r"([a-z]{2})([0-9A-Fa-f]+)", reply.split("er00/00", 1)[-1]))


if protocol_mode != "execution":
  raise SystemExit("nothing to read: a simulated machine has no configuration to rebuild")

machine = await star.driver.send_command(module="C0", command="RM")
extended = await star.driver.send_command(module="C0", command="QM")
read = {**read_fields(extended), **read_fields(machine)}

missing = [name for name in AK_PARAMETERS if name not in read]
print(f"read {len(AK_PARAMETERS) - len(missing)} of {len(AK_PARAMETERS)} parameters")
if missing:
  print(f"MISSING, so a safe write is not possible: {missing}")
else:
  as_it_stands = "".join(f"{name}{read[name]}" for name in AK_PARAMETERS)
  print(f"\nrestores exactly what the machine says now:\n  C0AK{as_it_stands}")

  with_monitoring = dict(read)
  with_monitoring["kb"] = f"{int(read['kb'], 16) | 0b100:02X}"
  proposed = "".join(f"{name}{with_monitoring[name]}" for name in AK_PARAMETERS)
  print(f"\nwith the front cover monitoring bit set:\n  C0AK{proposed}")
  print(f"\nkb {read['kb']} -> {with_monitoring['kb']}")
  print(
    "everything else identical:",
    as_it_stands.replace(f"kb{read['kb']}", "")
    == proposed.replace(f"kb{with_monitoring['kb']}", ""),
  )

2026-08-26 18:38:03,538 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RMid0144'
2026-08-26 18:38:03,606 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RMid0144er00/00kb0Bkp08 C00000 X00000 P10000 P20000 P30000 P40000 P50000 P60000 P70000 P80000 I00000 R00000 H00000')
2026-08-26 18:38:03,608 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0QMid0145'
2026-08-26 18:38:03,646 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0QMid0145er00/00ka010003xt54xa54xw13400xl07xr00xm03500xx11400ys090xu3540xv3700yu0060kl360kc0yx0060ke00000000xn00xo00ym6065kr0km360')


read 21 of 21 parameters

restores exactly what the machine says now:
  C0AKka010003ke00000000xt54xa54xw13400kb0Bxl07xn00xr00xo00xm03500xx11400xu3540xv3700kp08ys090kl360km360ym6065yu0060yx0060

with the front cover monitoring bit set:
  C0AKka010003ke00000000xt54xa54xw13400kb0Fxl07xn00xr00xo00xm03500xx11400xu3540xv3700kp08ys090kl360km360ym6065yu0060yx0060

kb 0B -> 0F
everything else identical: True


### Writing it

Only with `allow_configuration_write = True`, set in the cell itself so it cannot be reached by
running the notebook top to bottom. It writes the full command built above, reads the configuration
back, and prints the restore command in case the read-back does not match.

Keep the restore line from the cell above. If anything goes wrong, sending it puts the machine back.


In [22]:
allow_configuration_write = False

if not allow_configuration_write:
  print("skipped. this rewrites the machine's non-volatile configuration")
elif missing:
  print("refused: not every parameter could be read")
else:
  print(f"restore command, keep this:\n  C0AK{as_it_stands}\n")
  print(await star.driver.send_raw_command(f"C0AK{proposed}"))

  after = {
    **read_fields(await star.driver.send_command(module="C0", command="QM")),
    **read_fields(await star.driver.send_command(module="C0", command="RM")),
  }
  for name in AK_PARAMETERS:
    if after.get(name) != with_monitoring.get(name):
      print(f"  {name}: wrote {with_monitoring.get(name)}, reads back {after.get(name)}")
  print(
    "read back identical to what was written:",
    all(after.get(n) == with_monitoring.get(n) for n in AK_PARAMETERS),
  )

skipped. this rewrites the machine's non-volatile configuration


## 19- What the autoload holds in its own memory

Read-only, nothing moves. Four reads the driver did not have until now, each replacing something it
was assuming.

`request_module_configuration` is the one that matters. Its first field is the scanner's step size -
0.1 or 0.125 mm depending on the unit - which the driver used to hardcode; its second says whether
the loading indicators are fitted. Discovery reads both now, so this section checks that the read
works on hardware and that it agrees with what this machine was assumed to be. Last run answered
`au0 0 0 0 0`, so: 0.1 mm per step, indicators fitted.

The other three are diagnostic. `request_adjustment_status` says whether this autoload has ever been
adjusted - an unadjusted module holds factory defaults rather than its own values, and nothing
derived from them means much. `request_init_slot` gives the track the X drive homes against.
`request_adjustment_values` returns the whole adjustment block unparsed, because the 96-head's
equivalent came back truncated and the shape is worth seeing before anyone writes a parser.

`request_parameter` reads any of the 56 named parameters the module stores. A few worth having are
below; the full sweep belongs in the capture script, not here.


In [23]:
autoload = star.driver.autoload
if autoload is None:
  print("no autoload on this machine")
else:
  c = autoload.configuration

  # What discovery already read off this unit, and what the driver would have assumed without it.
  step, indicators = await autoload.request_module_configuration()
  print(f"scanner step      : {step} mm  (discovery stored {c.x_drive_mm_per_increment})")
  print(f"loading indicators: {'fitted' if indicators else 'none'}")
  if step != 0.1:
    print("  ! this unit is NOT the 0.1 mm generation - every autoload distance depended on that")

  adjusted_on, adjusted = await autoload.request_adjustment_status()
  print(f"adjusted          : {adjusted} on {adjusted_on}")
  if not adjusted:
    print("  ! unadjusted: its stored values are factory defaults, not this unit's")

  print(f"X drive homes at  : track {await autoload.request_init_slot()}")
  print(f"adjustment block  : {await autoload.request_adjustment_values()}")

  # Named parameters worth having: each drive's stored initialization position, and the barcode
  # reading geometry the carrier loads use.
  for name in ("kx", "ky", "kz", "bi", "bp", "bw", "cn", "co", "yl"):
    try:
      print(f"  {name} -> {await autoload.request_parameter(name)}")
    except Exception as e:  # noqa: BLE001 - a name this firmware does not know is an answer too
      print(f"  {name} -> refused: {e}")

2026-08-26 18:38:03,695 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RAid0146raau'
2026-08-26 18:38:03,709 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RAid0146au0 0 0 0 0')
2026-08-26 18:38:03,714 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RJid0147'
2026-08-26 18:38:03,724 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RJid0147jd2025-06-11js1')
2026-08-26 18:38:03,728 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0QXid0148'


scanner step      : 0.1 mm  (discovery stored 0.1)
loading indicators: fitted
adjusted          : True on 2025-06-11


2026-08-26 18:38:03,737 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0QXid0148bx01')
2026-08-26 18:38:03,741 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RKid0149'
2026-08-26 18:38:03,775 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RKid0149kx+055ky-065 +8921kz-211xx+0020 +025 +0400 +030 +0800 +042 +3000 +107yx+0020 +025 +0400 +030 +0800 +042 +3000 +107zx+0020 +043 +0100 +043 +0400 +051 +1500 +118')
2026-08-26 18:38:03,779 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RAid0150rakx'
2026-08-26 18:38:03,788 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RAid0150kx+055')
2026-08-26 18:38:03,790 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RAid0151raky'
2026-08-26 18:38:03,801 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RAid0151ky-065 +8921')
2026-08-26 18:38:03,804 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RAid0152rakz'
2026-08-26 18:38:03,814 - pylabrobot.io

X drive homes at  : track 1
adjustment block  : I0RKid0149kx+055ky-065 +8921kz-211xx+0020 +025 +0400 +030 +0800 +042 +3000 +107yx+0020 +025 +0400 +030 +0800 +042 +3000 +107zx+0020 +043 +0100 +043 +0400 +051 +1500 +118
  kx -> I0RAid0150kx+055
  ky -> I0RAid0151ky-065 +8921
  kz -> I0RAid0152kz-211
  bi -> I0RAid0153bi0067
  bp -> I0RAid0154bp0273


2026-08-26 18:38:03,855 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RAid0156racn'
2026-08-26 18:38:03,865 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RAid0156cn32')
2026-08-26 18:38:03,868 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RAid0157raco'


  bw -> I0RAid0155bw0132
  cn -> I0RAid0156cn32


2026-08-26 18:38:03,878 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RAid0157co0234')
2026-08-26 18:38:03,881 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RAid0158rayl'
2026-08-26 18:38:03,892 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RAid0158yl2000')


  co -> I0RAid0157co0234
  yl -> I0RAid0158yl2000


## 20- Does `C0 QA` really answer 0 between tracks?

`request_track` is documented as returning the track the carrier handler sits at, "or 0 when it is
at neither end of a track". Nothing has ever checked that, and two things depend on it: a caller
using 0 to mean "between tracks" rather than "track 0", and the simulator, which cannot produce a
0 at all because it answers from a track it was last told to go to.

The sled is moved to a track, then to a position between two tracks, and `QA` is asked at each. If
the sentence holds, the second read answers 0. The last probes walk away from a track in 2 mm steps
to find how wide the band is that still reports a track - which is the number a caller needs if it
ever converts a track to a position.

**This moves the autoload sled** along the front of the deck, which is what `move_to_track` does
during any carrier load. The wheel is raised first, by the same guard those loads use. Gated on
`allow_autoload_move`, set in the cell itself so running the notebook top to bottom does not move
anything.


In [24]:
allow_autoload_move = True

if not allow_autoload_move:
  print("skipped. set allow_autoload_move = True to move the sled")
elif star.driver.autoload is None:
  print("no autoload on this machine")
else:
  autoload = star.driver.autoload
  track = 10
  on_track = deck.rails_to_location(track).x
  pitch = deck.rails_to_location(track + 1).x - on_track
  print(f"track {track} sits at {on_track} mm, tracks are {pitch} mm apart\n")

  await autoload.move_to_track(track)
  print(
    f"  on track {track:2d}      : QA says {await autoload.request_track():2d}, "
    f"drive at {await autoload.request_x_position():7.2f} mm"
  )

  # Halfway to the next one, which is as far from either as it is possible to be.
  await autoload.move_x(on_track + pitch / 2)
  between = await autoload.request_track()
  print(
    f"  halfway to {track + 1:2d}    : QA says {between:2d}, "
    f"drive at {await autoload.request_x_position():7.2f} mm"
  )
  print(f"\n  the documented 0 between tracks: {'holds' if between == 0 else 'DOES NOT HOLD'}\n")

  # How far off a track it can be and still be reported as on it.
  for offset in (2.0, 4.0, 6.0, 8.0, 10.0):
    await autoload.move_x(on_track + offset)
    reported = await autoload.request_track()
    print(f"  {offset:4.1f} mm past {track:2d}: QA says {reported:2d}")
    if reported != track:
      print(f"  -> still reported as track {track} up to somewhere under {offset} mm past it")
      break

  await autoload.park()
  print(
    f"\n  parked; QA says {await autoload.request_track()}, "
    f"drive at {await autoload.request_x_position():.2f} mm"
  )

2026-08-26 18:38:03,907 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RZid0159'


track 10 sits at 302.5 mm, tracks are 22.5 mm apart



2026-08-26 18:38:03,918 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RZid0159rz+0000 +0002')
2026-08-26 18:38:03,922 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0XPid0160xp10xv2500xr3xw7'
2026-08-26 18:38:08,282 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0XPid0160er00')
2026-08-26 18:38:08,287 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RXid0161'
2026-08-26 18:38:08,297 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RXid0161rx+02025 +02025')
2026-08-26 18:38:08,301 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0QAid0162'
2026-08-26 18:38:08,331 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0QAid0162er00/00qa10')
2026-08-26 18:38:08,336 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RXid0163'
2026-08-26 18:38:08,345 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RXid0163rx+02025 +02025')
2026-08-26 18:38:08,350 - pylabrobot.io.usb - IO - [0x8af

  on track 10      : QA says 10, drive at  302.50 mm


2026-08-26 18:38:08,671 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0XAid0165er00')
2026-08-26 18:38:08,676 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RXid0166'
2026-08-26 18:38:08,685 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RXid0166rx+02138 +02138')
2026-08-26 18:38:08,689 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0QAid0167'
2026-08-26 18:38:08,721 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0QAid0167er00/00qa11')
2026-08-26 18:38:08,726 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RXid0168'
2026-08-26 18:38:08,735 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RXid0168rx+02138 +02138')
2026-08-26 18:38:08,740 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RZid0169'
2026-08-26 18:38:08,749 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RZid0169rz+0000 +0002')
2026-08-26 18:38:08,752 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] wri

...

   8.0 mm past 10: QA says 10


2026-08-26 18:38:09,960 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0QAid0188er00/00qa10')
2026-08-26 18:38:09,964 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RZid0189'
2026-08-26 18:38:09,974 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RZid0189rz+0000 +0002')
2026-08-26 18:38:09,977 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0XPid0190xp54xv2500xr3xw7'


  10.0 mm past 10: QA says 10


2026-08-26 18:38:14,285 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0XPid0190er00')
2026-08-26 18:38:14,288 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RXid0191'
2026-08-26 18:38:14,298 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RXid0191rx+11925 +11924')
2026-08-26 18:38:14,300 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0QAid0192'
2026-08-26 18:38:14,332 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0QAid0192er00/00qa54')
2026-08-26 18:38:14,334 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'I0RXid0193'
2026-08-26 18:38:14,346 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'I0RXid0193rx+11925 +11924')



  parked; QA says 54, drive at 1292.40 mm


## 21- Raw command escape hatch

Anything not yet wrapped in a named method can be sent directly. **Only send commands you have
confirmed are read-only** - this bypasses every guard in the driver.

In [25]:
# C0 RF - request the master's firmware version. Read-only.
print(await star.driver.send_command(module="C0", command="RF"))

# the same thing as a raw string, id included
# print(await star.driver.send_raw_command("C0RFid9999"))

2026-08-26 18:38:14,356 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RFid0194'
2026-08-26 18:38:14,371 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RFid0194er00/00rf7.6S 25 2021_11_05 (GRU C0)')


C0RFid0194er00/00rf7.6S 25 2021_11_05 (GRU C0)


## 22- Teardown


In [26]:
await star.stop()
print("disconnected. connected:", star.driver.connected)

# The log is append-only and stays open for the rest of the session - nothing to close.

2026-08-26 18:38:14,410 - pylabrobot.io.usb - WARNING - Closing connection to USB device.


disconnected. connected: False
